In [22]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.ensemble import GradientBoostingRegressor

In [12]:
data = pd.read_csv("../data/daily_restaurant_sales.csv")
data.head()

,rest_id,day,month,weekday,weekend,is_holiday,is_dec,is_nov,competitors_price,discounts,sales
0,0,2016-01-01,1,4,False,True,False,False,2.88,0,79.0
1,0,2016-01-02,1,5,True,False,False,False,2.64,0,57.0
2,0,2016-01-03,1,6,True,False,False,False,2.08,5,294.0
3,0,2016-01-04,1,0,False,False,False,False,3.37,15,676.5
4,0,2016-01-05,1,1,False,False,False,False,3.79,0,66.0


In [15]:
X = ["C(month)", "C(weekday)","is_holiday", "competitors_price"] # 공변량 정보들
regr_cate = smf.ols(f"sales ~ discounts*({'+'.join(X)})", data = data).fit() # 모든 공변량 * 처치 상호작용 term 적용

In [ ]:
ols_cate_pred = (
    regr_cate.predict(data.assign(discounts=data["discounts"] + 1))
    - regr_cate.predict(data)
)

In [21]:
train = data.query("day<'2018-01-01'")
test = data.query("day>='2018-0101'")

X = ["C(month)", "C(weekday)", "is_holiday", "competitors_price"]
regr_model = smf.ols(f"sales ~ discounts*({'+'.join(X)})", data=train).fit()

cate_pred = (
    regr_model.predict(test.assign(discounts=test["discounts"]+1)) # 처치 + 1 
    -regr_model.predict(test)
)

In [25]:
X = ["month", "weekday", "is_holiday", "competitors_price", "discounts"]
y = "sales"

np.random.seed(1)
ml_model = GradientBoostingRegressor(n_estimators = 50).fit(train[X],train[y])

ml_pred = ml_model.predict(test[X])

In [43]:
np.random.seed(123)

test_pred = test.assign(
    ml_pred=ml_pred,
    cate_pred=cate_pred,
    rand_m_pred=np.random.uniform(-1, 1, len(test)),
)

In [29]:
# 커리 데코레이터
from toolz import curry

@curry
def effect(data, y, t):
    return (np.sum((data[t] - data[t].mean())*data[y]) /
            np.sum((data[t] - data[t].mean())**2))


effect(test, "sales", "discounts")

32.0251752116039

In [32]:
def effect_by_quantile(df, pred, y , t, q=10):

    # 분위수에 대한 파티션 생성
    groups = np.round(pd.IntervalIndex(pd.qcut(df[pred], q=q)).mid, 2)
                      
    return (df
            .assign(**{f"{pred}_quantile": groups})
            .groupby(f"{pred}_quantile")
            .apply(effect(y=y, t=t)))

effect_by_quantile(test_pred, "cate_pred", y="sales", t="discounts")

C:\Users\LEEBG\AppData\Local\Temp\ipykernel_34268\825019389.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(effect(y=y, t=t)))


cate_pred_quantile
17.20    19.757169
23.38    25.058695
26.37    26.591238
28.47    28.627493
30.35    29.451661
32.17    30.804326
33.97    35.028346
35.99    36.682979
38.70    38.752834
45.17    43.794884
dtype: float64

In [33]:
def cumulative_effect_curve(dataset, prediction, y, t,
                            ascending=False, steps=100):
    size = len(dataset)
    ordered_df = (dataset
                  .sort_values(prediction, ascending=ascending)
                  .reset_index(drop=True))
    
    steps = np.linspace(size/steps, size, steps).round(0)
    
    return np.array([effect(ordered_df.query(f"index<={row}"), t=t, y=y)
                     for row in steps])

cumulative_effect_curve(test_pred, "cate_pred", "sales", "discounts")

array([49.01939967, 45.51485777, 45.44213755, 44.85148871, 44.5564373 ,
       44.50137191, 44.34909372, 44.2569066 , 44.01518992, 43.79488364,
       43.39785123, 43.05421385, 42.34574731, 42.16711703, 41.88298069,
       41.62112688, 41.54072967, 41.20221333, 41.13893991, 40.96368272,
       40.70175089, 40.51960457, 40.15809843, 40.04559321, 39.97222001,
       39.71969922, 39.53614293, 39.53469557, 39.43161658, 39.30138393,
       39.24237582, 39.23720113, 39.07583734, 38.9974004 , 38.91120733,
       38.85559696, 38.70085068, 38.67853266, 38.49645046, 38.51792424,
       38.33597876, 38.23810769, 37.99579126, 37.93259754, 37.89290761,
       37.70306323, 37.51707738, 37.29396239, 37.21900933, 37.1637756 ,
       36.98490646, 36.92462992, 36.88588775, 36.70289459, 36.40991886,
       36.36245281, 36.33487701, 36.17031981, 36.08349811, 35.83274796,
       35.69633073, 35.49466022, 35.34656124, 35.25980869, 35.20611859,
       35.13418348, 35.11626019, 35.01793434, 34.96925453, 34.87

In [34]:
def cumulative_gain_curve(df, prediction, y, t,
                          ascending=False, normalize=False, steps=100):
    
    effect_fn = effect(t=t, y=y)
    normalizer = effect_fn(df) if normalize else 0
    
    size = len(df)
    ordered_df = (df
                  .sort_values(prediction, ascending=ascending)
                  .reset_index(drop=True))
    
    steps = np.linspace(size/steps, size, steps).round(0)
    effects = [(effect_fn(ordered_df.query(f"index<={row}"))
                -normalizer)*(row/size) 
               for row in steps]

    return np.array([0] + effects)


cumulative_gain_curve(test_pred, "cate_pred", "sales", "discounts")

array([ 0.        ,  0.48078729,  0.91223809,  1.35648172,  1.79788484,
        2.22307171,  2.67577545,  3.10159973,  3.54810168,  3.9604286 ,
        4.37015044,  4.77468896,  5.15916166,  5.50765583,  5.89800187,
        6.28691224,  6.65583052,  7.06812415,  7.41464137,  7.82429305,
        8.19273654,  8.53955712,  8.91604092,  9.23036889,  9.61435777,
        9.98879357, 10.33220321, 10.67222963, 11.07645842, 11.43432805,
       11.798795  , 12.16597323, 12.54921145, 12.89752584, 13.25412713,
       13.62307088, 13.98470099, 14.325091  , 14.696193  , 15.02100306,
       15.40716969, 15.71039471, 16.06163585, 16.33251923, 16.69357811,
       17.04776867, 17.34823251, 17.63062656, 17.90746339, 18.23652099,
       18.58981185, 18.86309089, 19.19450912, 19.55187994, 19.81486761,
       20.02933703, 20.3598723 , 20.71630301, 20.97724305, 21.29618822,
       21.49964878, 21.7679117 , 22.00820297, 22.26305798, 22.5692848 ,
       22.88022377, 23.19305588, 23.52564809, 23.81816856, 24.12

In [42]:
X = ["C(month)", "C(weekday)", "is_holiday", "competitors_price"]

y_res = smf.ols(f"sales ~ {'+'.join(X)}", data=test).fit().resid
t_res = smf.ols(f"discounts ~ {'+'.join(X)}", data=test).fit().resid

tau_hat = y_res/t_res

In [44]:
from sklearn.metrics import mean_squared_error

for m in ["rand_m_pred", "ml_pred", "cate_pred"]:
    wmse = mean_squared_error(tau_hat, test_pred[m],
                              sample_weight=t_res**2)
    print(f"MSE for {m}:", wmse)

MSE for rand_m_pred: 1097.3201917848464
MSE for ml_pred: 575434.4080564725
MSE for cate_pred: 43.46750050123515
